# Week 5 - Prediksi karakter dengan GRU
Marvel Kevin Nathanael | 00000108042

Studi kasus text generation TLM 5: jendela karakter, target digeser satu posisi, dan sampling dengan temperature. Corpus Alice diunduh saat run.

In [1]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
url = "https://www.gutenberg.org/cache/epub/11/pg11.txt"
file_path = tf.keras.utils.get_file("alice_in_wonderland.txt", origin=url)
text = open(file_path, encoding="utf-8-sig").read()[:80_000]
cut = int(len(text) * 0.8)
train_text, val_text = text[:cut], text[cut:]
vocab = sorted(set(train_text))
lookup = tf.keras.layers.StringLookup(vocabulary=vocab, mask_token=None)
inverse = tf.keras.layers.StringLookup(
    vocabulary=lookup.get_vocabulary(), invert=True, mask_token=None)
def encode(s):
    return lookup(tf.strings.unicode_split(s, "UTF-8")).numpy().astype("int32")
train_ids, val_ids = encode(train_text), encode(val_text)
print("Karakter train/val:", len(train_ids), len(val_ids),
      "| Vocab:", lookup.vocabulary_size())

Karakter train/val: 64000 16000 | Vocab: 85


In [2]:
SEQ_LEN = 60
def make_dataset(ids, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(ids)
    ds = ds.batch(SEQ_LEN + 1, drop_remainder=True)
    ds = ds.map(lambda chunk: (chunk[:-1], chunk[1:]),
                num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=42)
    return ds.batch(64).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_ids, shuffle=True)
val_ds = make_dataset(val_ids)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(None,), dtype=tf.int32),
    tf.keras.layers.Embedding(lookup.vocabulary_size(), 32),
    tf.keras.layers.GRU(64, return_sequences=True),
    tf.keras.layers.Dense(lookup.vocabulary_size())
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["sparse_categorical_accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=3, verbose=0, shuffle=False)
print("Validation loss terakhir:", round(history.history["val_loss"][-1], 3))

Validation loss terakhir: 2.915


In [3]:
def generate(prefix="Alice", n_chars=120, temperature=0.8):
    ids = encode(prefix).tolist()
    if any(i == 0 for i in ids):
        raise ValueError("Prefix berisi karakter di luar vocab train")
    for _ in range(n_chars):
        logits = model(tf.constant([ids[-SEQ_LEN:]], dtype=tf.int32),
                       training=False)[0, -1]
        next_id = int(tf.random.categorical(logits[None, :] / temperature, 1)[0, 0])
        ids.append(next_id)
    return "".join(x.decode("utf-8") for x in inverse(tf.constant(ids)).numpy())

print(generate("Alice", temperature=0.8))

Alice ittee an me suti
lian wn ld int mir ngertsoy
k ghhsidy be sly oty to,ot ons ane t cssse a*ia sio tea
he py whh.ahoy ibr


**Catatan:** Split berdasarkan posisi teks dibuat sebelum window agar window bertumpuk tidak bocor ke validasi. Tiga epoch pada sampel korpus hanya demonstrasi; hasil teks bisa belum koheren.